# Voice AI Fundamentals

A voice agent turns audio into text or model input, decides what to say, produces audio, and manages conversation timing.


## What to learn

- Speech recognition converts audio into text in a cascaded system.
- Speech generation converts the response into audio.
- Voice activity and end-of-turn detection decide when the user has finished speaking.
- Barge-in stops playback when the user interrupts.

## Decision rule

Optimize the complete conversational loop, not one model in isolation. Stream each stage, measure latency percentiles, preserve interruption state, handle silence and noise explicitly, and disclose recording and data use.


## Runnable example

Run the following cells, then change one input and observe what changes. Focus on the decision being demonstrated rather than framework-specific syntax.


In [ ]:
"""Simulate a voice turn's latency budget and barge-in truncation logic.

Pure simulation: shows the two calculations interviewers ask you to reason
about — (1) where does the second go in a voice turn, (2) how do you truncate
history when the user interrupts mid-sentence.
"""

# --- 1. Latency budget ---------------------------------------------------
STAGES = [
    ("endpoint wait (silence/semantic)", 250),
    ("STT final transcript", 180),
    ("LLM time-to-first-token", 450),
    ("TTS time-to-first-byte", 90),
    ("network + jitter buffer", 60),
]

total = sum(ms for _, ms in STAGES)
print(f"voice-to-voice: {total}ms  (target: <800ms P95)")
for name, ms in STAGES:
    bar = "#" * (ms // 20)
    print(f"  {name:<36} {ms:>4}ms {bar}")

# --- 2. Barge-in truncation ----------------------------------------------
# Define the data shape and small operations before running them.
def truncate_on_interrupt(agent_text: str, word_timestamps: list[tuple[str, int]],
                          interrupted_at_ms: int) -> str:
    """Keep only the words the user actually heard before interrupting.

    word_timestamps: (word, start_ms) pairs from the TTS engine.
    Without this, the chat history claims the agent said things the user
    never heard, and the conversation derails.
    """
    heard = [w for w, start in word_timestamps if start < interrupted_at_ms]
    return " ".join(heard)


tts_words = [("Thanks", 0), ("for", 200), ("sharing", 350), ("that.", 600),
             ("Now", 900), ("let's", 1050), ("talk", 1200), ("about", 1350),
             ("pricing.", 1500)]
history_entry = truncate_on_interrupt("...", tts_words, interrupted_at_ms=1000)
print("\nagent history after barge-in at t=1000ms:")
print(f'  "{history_entry}"  (not the full scripted sentence)')


## Study questions

1. What problem does this pattern solve?
2. What is the simplest alternative?
3. Which failure would tell you that the pattern is implemented incorrectly?
4. What measurement would justify adding more complexity?

## Before using it

- [ ] The requirement and success measure are explicit.
- [ ] Inputs and outputs are validated.
- [ ] Failures, retries, and stopping conditions are bounded.
- [ ] Security and privacy match the consequences of the action.
- [ ] The behavior is tested on realistic examples.
